# Text-to-Image Generation — Setup and Training

**Before running:** Runtime → Change runtime type → T4 GPU

**Run cells in order. Do NOT skip any cell.**

This notebook:
1. Mounts Google Drive (saves checkpoints so they survive session disconnects)
2. Clones the repo
3. Installs dependencies
4. Downloads the dataset
5. Generates captions with BLIP
6. Prepares the dataset
7. Trains LoRA (rank 16, 2000 steps)
8. Saves everything to Google Drive

## Step 1 — Mount Google Drive

Everything gets saved here so it survives session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/t2i_project'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Saving everything to: {DRIVE_DIR}')

## Step 2 — Check GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## Step 3 — Clone Repo or Resume from Drive

If you've run this notebook before, the repo already exists on Drive — we copy it back instead of cloning fresh.

In [ ]:
import os, shutil

REPO_ON_DRIVE = f'{DRIVE_DIR}/Text-To-Image-Generation'
REPO_LOCAL   = '/content/Text-To-Image-Generation'

if os.path.exists(REPO_ON_DRIVE):
    print('Resuming from Drive...')
    if os.path.exists(REPO_LOCAL):
        shutil.rmtree(REPO_LOCAL)
    shutil.copytree(REPO_ON_DRIVE, REPO_LOCAL)
    print('Copied repo from Drive to Colab.')
else:
    print('Fresh clone from GitHub...')
    !git clone https://github.com/snyadav-ds-ml/Text-To-Image-Generation.git
    print('Cloned.')

%cd /content/Text-To-Image-Generation
!ls

## Step 4 — Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## Step 5 — Download Dataset

Skips download if already on Drive.

In [ ]:
import os

RAW_DIR = 'data/raw'
os.makedirs(RAW_DIR, exist_ok=True)

# images
if not os.path.exists(f'{RAW_DIR}/jpg'):
    print('Downloading flower images (~330MB)...')
    !wget -q https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz -P {RAW_DIR}
    !tar -xzf {RAW_DIR}/102flowers.tgz -C {RAW_DIR}
    print('Images extracted.')
else:
    print('Images already downloaded.')

# split file
if not os.path.exists(f'{RAW_DIR}/setid.mat'):
    !wget -q https://www.robots.ox.ac.uk/~vgg/data/flowers/102/setid.mat -P {RAW_DIR}
    print('setid.mat downloaded.')

# labels file
if not os.path.exists(f'{RAW_DIR}/imagelabels.mat'):
    !curl -sL -o {RAW_DIR}/imagelabels.mat https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat
    print('imagelabels.mat downloaded.')

print('\ndata/raw contents:')
!ls -lh data/raw/

## Step 6 — Generate Captions with BLIP

Runs BLIP on all 8,189 images. Takes ~30 min on T4.
Resumable — skips already-captioned images.

In [ ]:
!python src/data/generate_captions.py

## Step 7 — Prepare Dataset

Resizes images to 512×512, writes metadata.jsonl and split files.

In [ ]:
!python src/data/prepare_dataset.py

# verify output
import os
for split in ['train', 'val', 'test']:
    n = len(os.listdir(f'data/processed/images/{split}'))
    print(f'{split}: {n} images')

## Step 8 — Save Dataset to Drive

Save processed data to Drive before training starts — in case session disconnects.

In [ ]:
import shutil, os

# save captions
dst_captions = f'{DRIVE_DIR}/data_raw_captions'
if not os.path.exists(dst_captions):
    shutil.copytree('data/raw/captions', dst_captions)
    print('Captions saved to Drive.')
else:
    print('Captions already on Drive.')

# save processed metadata
dst_metadata = f'{DRIVE_DIR}/metadata.jsonl'
shutil.copy('data/processed/metadata.jsonl', dst_metadata)
print('metadata.jsonl saved to Drive.')

## Step 9 — Train LoRA

Trains SD v1.5 with LoRA (rank 16) for 2,000 steps.
Takes ~2-3 hours on T4.

Checkpoints are saved every 500 steps to `outputs/checkpoints/main_lora_r16/`.

In [ ]:
!python src/train/train_lora.py --config configs/train_lora_r16.yaml

## Step 10 — Save Checkpoints to Drive

**Run this immediately after training.** Don't wait — Colab sessions disconnect without warning.

In [ ]:
import shutil, os

dst = f'{DRIVE_DIR}/checkpoints'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree('outputs/checkpoints', dst)
print(f'Checkpoints saved to Drive: {dst}')
!ls {dst}/main_lora_r16/

## Step 11 — Quick Test: Generate 5 Images

Sanity check that the LoRA weights work before running the full 1,000-image evaluation.

In [ ]:
import sys
sys.path.insert(0, '.')

from src.models.load_pipeline import load_pipeline
from src.utils.seed import set_seed
import matplotlib.pyplot as plt

TEST_PROMPTS = [
    'a bright yellow sunflower with a dark center',
    'a pink peony in full bloom',
    'a red rose with water droplets',
    'a purple iris with detailed veins',
    'a white daisy on a green meadow',
]

# load fine-tuned model
pipe = load_pipeline(
    'runwayml/stable-diffusion-v1-5',
    lora_path='outputs/checkpoints/main_lora_r16/final'
)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, prompt in enumerate(TEST_PROMPTS):
    set_seed(42)
    image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
    axes[i].imshow(image)
    axes[i].set_title(prompt[:30] + '...', fontsize=7)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('outputs/quick_test.png', dpi=150)
plt.show()
print('Saved to outputs/quick_test.png')

## Step 12 — Save Full Repo to Drive

Saves the entire project (code + outputs) to Drive.

In [ ]:
import shutil, os

dst = REPO_ON_DRIVE
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree('/content/Text-To-Image-Generation', dst)
print(f'Full repo saved to Drive: {dst}')